# nb05 — финальный конвейер извлечения: таблицы + номера + формулы

Цель: целостный прототип, достающий из нормативного PDF **таблицы (содержимое + номер)** и
**формулы (исполняемый LaTeX → sympy)** для обозначений любого алфавита (кириллица,
греческий, латиница). Пилот — `СП 31.13330.2021` (160 стр., text-based).

Подход: слоистый. Layout (DocLayout-YOLO) сегментирует блоки; содержимое берётся из
текстового слоя (pdfplumber), формулы распознаёт pix2text, индексы-обозначения
восстанавливаются по словарю `extract_formula_notation`.

## 0. Окружение и модели

In [1]:
from pathlib import Path
import re
from collections import Counter, defaultdict

import numpy as np
import pdfplumber
from doclayout_yolo import YOLOv10
from huggingface_hub import hf_hub_download

from normatives_pipeline.formulas import detect
from normatives_pipeline.formulas.notation_restore import restore_indices
from normatives_pipeline.formulas.validation import canonical_latex
from normatives_pipeline.parser import extractors, text_layer

ROOT = Path.cwd()
if not (ROOT / "docs" / "normative-base").exists():
    ROOT = Path.home() / "CalcEngine"
PDF = ROOT / "docs" / "normative-base" / "СП 31.13330.2021.pdf"
DPI = 400
SCALE = DPI / 72

weights = hf_hub_download(
    "juliozhao/DocLayout-YOLO-DocStructBench",
    "doclayout_yolo_docstructbench_imgsz1024.pt",
)
layout_model = YOLOv10(weights)
print("layout model ready")

layout model ready


## 1. Таблицы: содержимое (pdfplumber) + номер (table_caption)

`pdfplumber.find_tables` даёт строки; номер берём из `table_caption` (DocLayout-YOLO).
Подпись привязываем по геометрии (выше таблицы, пересекается по горизонтали, ближайшая по
вертикали). Номер извлекаем после удаления разрядки (`"Т а б л и ц а 8"` → `"Таблица8"`).
Для продолжения таблицы на следующей странице (caption нет) наследуем последний номер.

In [2]:
def layout_blocks(img_bgr):
    blocks = defaultdict(list)
    for r in layout_model.predict(img_bgr, conf=0.25, imgsz=1024, verbose=False):
        for box, cls in zip(r.boxes.xyxy, r.boxes.cls):
            name = r.names[int(cls)]
            if name in ("table", "table_caption", "isolate_formula", "formula_caption"):
                blocks[name].append(tuple(int(v) for v in box))
    return blocks


def caption_text(page, bbox):
    x0, y0, x1, y1 = (v / SCALE for v in bbox)
    return page.crop((x0, y0, x1, y1)).extract_text() or ""


def table_number_from_caption(page, bbox):
    nos = re.sub(r"\s+", "", caption_text(page, bbox))
    m = re.search(r"[Тт]аблица(\d+(?:\.\d+)?)", nos)
    return m.group(1) if m else ""


def extract_tables(pdf):
    raw = []
    for i, page in enumerate(pdf.pages, 1):
        img = np.array(page.to_image(resolution=DPI).original)
        blocks = layout_blocks(img[..., ::-1])
        for tb in blocks["table"]:
            cand = [c for c in blocks["table_caption"] if c[3] <= tb[1] and c[0] < tb[2] and c[2] > tb[0]]
            num = table_number_from_caption(page, max(cand, key=lambda c: c[3])) if cand else ""
            raw.append({"page": i, "number": num})
    last = ""
    for t in raw:
        if not t["number"]:
            t["number"] = last
        else:
            last = t["number"]
    return raw

In [3]:
pdf = pdfplumber.open(PDF)
tables = extract_tables(pdf)
pdf.close()

numbered = [t for t in tables if t["number"]]
print(f"layout table blocks: {len(tables)}")
print(f"with number (caption + inherit): {len(numbered)} ({len(numbered)/len(tables):.0%})")
print("distinct numbers:", len({t['number'] for t in tables if t['number']}))

layout table blocks: 69
with number (caption + inherit): 69 (100%)
distinct numbers: 22


## 2. Формулы: детекция (layout) → распознавание (pix2text) → восстановление → sympy

Обходим **все** страницы layout-детекцией. Номер формулы читаем из `formula_caption`
справа от блока; по номеру берём словарь обозначений и восстанавливаем кириллические
индексы. Поверх `correct_latex` применяем `harden` (OCR-артефакты: `\Sigma`→`\sum`,
слитное `\sumx`→`\sum x`, `\tt`).

In [4]:
pages_text = text_layer.read_pages(PDF)
regex_formulas = extractors.extract_formulas(pages_text)
notation = extractors.extract_formula_notation(pages_text, regex_formulas)


def classify_notation(symbol):
    if any("\u0400" <= ch <= "\u04ff" for ch in symbol):
        return "cyr"
    if any("\u0370" <= ch <= "\u03ff" for ch in symbol):
        return "greek"
    return "lat"


def formula_number_from_caption(page, bbox, captions):
    for cb in captions:
        if cb[1] < bbox[3] and bbox[1] < cb[3] and cb[0] >= bbox[2]:
            nos = re.sub(r"\s+", "", caption_text(page, cb))
            m = re.search(r"\((\d+(?:\.\d+)?)\)", nos)
            if m:
                return m.group(1)
    return ""


def harden(latex: str) -> str:
    latex = re.sub(r"\\sum(?!_)", lambda m: "\\Sigma", latex)
    latex = re.sub(r"\\Sigma([A-Za-z])", lambda m: "\\Sigma " + m.group(1), latex)
    latex = re.sub(r"\\tt(?![a-zA-Z])", "", latex)
    latex = latex.replace("\\texttt", "\\text")
    latex = latex.replace("{,}", ".")
    return latex


def process_formulas(pdf):
    results = []
    for i, page in enumerate(pdf.pages, 1):
        img = np.array(page.to_image(resolution=DPI).original)
        blocks = layout_blocks(img[..., ::-1])
        for bbox in blocks["isolate_formula"]:
            num = formula_number_from_caption(page, bbox, blocks["formula_caption"])
            crop = img[bbox[1]:bbox[3], bbox[0]:bbox[2]]
            raw = detect.recognize_crop(crop[..., ::-1])
            symbols = [e.symbol for e in notation.get(num, [])] if num else []
            restored = harden(restore_indices(raw, symbols) if symbols else raw)
            parsed = not isinstance(canonical_latex(restored), str)
            kinds = Counter(classify_notation(s) for s in symbols)
            results.append({"page": i, "number": num, "raw": raw,
                            "restored": restored, "parsed": parsed, "kinds": dict(kinds)})
    return results

In [5]:
pdf = pdfplumber.open(PDF)
results = process_formulas(pdf)
pdf.close()

parsed = [r for r in results if r["parsed"]]
print(f"isolate_formula blocks: {len(results)}")
print(f"parsed by latex2sympy2: {len(parsed)} ({len(parsed)/len(results):.0%})")
print()
for r in results:
    flag = "OK  " if r["parsed"] else "FAIL"
    print(f"{flag} p{r['page']:>3} f{r['number'] or '?':>4} kinds={r['kinds']} | {r['restored']}")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Could not find any ONNX files with standard file name decoder_model_merged.onnx, files found: [PosixPath('encoder_model.onnx'), PosixPath('decoder_model.onnx')]. Please make sure to pass a `file_name` and/or `subfolder` argument to `from_pretrained` when loading an ONNX file with non-standard file names.


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:03<00:00,  3.63s/it]

100%|██████████| 1/1 [00:03<00:00,  3.63s/it]

  0%|          | 0/1 [00:00<?, ?it/s]

... (56 outputs left out)

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:02<00:00,  2.52s/it]

100%|██████████| 1/1 [00:02<00:00,  2.52s/it]

isolate_formula blocks: 21
parsed by latex2sympy2: 11 (52%)

OK   p 12 f   1 kinds={} | Q_{_{\mathrm{ K}}}=\Sigma q_{_{\mathrm{ K}}} N_{_{\mathrm{ K}}} / 1 0 0 0
FAIL p 13 f   2 kinds={} | \left. \begin{aligned} & { Q_{\mathrm{ c y r . m a x}}=K_{\mathrm{ c y r . m a x}} Q_{\mathrm{ c y r . m}} ;} \ & { Q_{\mathrm{ c y r . m i n}}=K_{\mathrm{ c y r . m i n}} Q_{\mathrm{ c y r . m}} .} \ \end{aligned} \right\}
FAIL p 13 f   4 kinds={'greek': 1} | .\begin{aligned}&{K_{{q.max}}=\alpha_{max}\beta_{max};}\&{K_{{q.min}}=\alpha_{min}\beta_{min}.}\\end{aligned}\}
FAIL p 13 f   3 kinds={'greek': 1} | .\begin{aligned}&{q_{q.max}=K_{q.max}Q_{cyr.max}/24;}\&{q_{q.min}=K_{q.min}Q_{cyr.min}/24.}\\end{aligned}\}
FAIL p 13 f   ? kinds={} | K_{\mathrm{c y r . m a x}}=1.1-1.3 ; K_{\mathrm{c y r . m i n}}=0.7-0.9
FAIL p 39 f   5 kinds={'cyr': 1} | \mu_{k}=K_{m(\frac{A_{\kappa}}{e_{\kappa}}-u_0)+1,}
OK   p 42 f   6 kinds={'cyr': 1} | h=\zetav^{2}/2g
FAIL p 44 f   7 kinds={'lat': 1} | F_{B.o}=q\beta_{06}/3

## 3. Выводы

Замер кириллицы по пяти СП (детерминированно, без моделей; `symbols.normalize_symbol`):

- Кириллица есть в обозначениях всех СП: 72 из 465 записей (≈15%).
- Отображение кириллицы корректно: `latex` = `\text{…}`, MathJax рендерит верно.
- Транслит `cyr_map` (а→a, с→s, е→e, р→r, у→u, к→k) **схлопывал латиницу и кириллицу
  в один `id`**: по извлечённым обозначениям было 8 групп коллизий, где одному `id`
  соответствовали разные `latex` (например `s` ← `s`, `\text{с}`, `\text{С}`); с учётом
  manifest-формул в БД — 11 групп.
- Решение (реализовано в `parser/symbols.py`): обозначению с кириллицей добавляется
  суффикс `_cyr`, а регистр сохраняется (`А`→`A_cyr`, `а`→`a_cyr`); латинское `s`→`s`.
  Кириллические коллизии после этого — **0** (остаются только форматные `_g` vs `_{g}`
  между manifest и извлечением — это отдельная задача нормализации LaTeX).
- Разбор формул (`restore_indices` + `latex2sympy2`): кириллические индексы транслитерируются
  для sympy, отображение остаётся кириллическим. Это закрывает формулы с блоком «где»;
  без словаря (вариант B/C в `RECOMMENDATION.md`) остаётся потолок ~48% на СП 31.

Аудит и миграция старых данных — `python -m normatives_pipeline.parser.cyrillic [--apply]`.


## 4. Кириллица: коллизии `id` и разбор (детерминированно)

Проверяем, остаётся ли `id` обозначения однозначным при кириллице. Считаем по текстовому
слою (без моделей): сколько обозначений кириллические/греческие/латинские и сколько `id`
имеют больше одного `latex` — «до» (старый транслит без суффикса) и «после» фикса
`_cyr` в `parser/symbols.py`.


In [6]:
from collections import defaultdict
import re
from pathlib import Path

from normatives_pipeline.formulas.notation_restore import restore_indices
from normatives_pipeline.formulas.validation import canonical_latex
from normatives_pipeline.parser import extractors, symbols, text_layer

ROOT = Path.cwd().resolve()
while not (ROOT / "docs" / "normative-base").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DOCS = ["СП 22.13330.2016", "СП 24.13330.2021", "СП 30.13330.2020",
        "СП 31.13330.2021", "СП 45.13330.2017"]


def notation_rows(pages):
    clauses = extractors.extract_clauses(pages)
    numbered = extractors.associate_formulas_to_clauses(
        clauses, extractors.extract_formulas(pages)
    )
    unnumbered, un_notation = extractors.extract_unnumbered_formulas(pages, clauses)
    notation = extractors.extract_formula_notation(pages, numbered)
    notation.update(un_notation)
    return numbered + unnumbered, notation


print(f"{'документ':22} {'обозн.':>7} {'кир':>5} {'греч':>5} {'лат':>5} {'колл.до':>8} {'колл.после':>11}")
tot = [0, 0, 0, 0, 0, 0]
for doc in DOCS:
    pages = text_layer.read_pages(ROOT / "docs" / "normative-base" / f"{doc}.pdf")
    formulas, notation = notation_rows(pages)
    old_g, new_g = defaultdict(set), defaultdict(set)
    cyr = greek = lat = 0
    for f in formulas:
        for n in notation.get(f.number, []):
            for raw in symbols.split_symbols(n.symbol):
                res = symbols.normalize_symbol(raw)
                if res is None:
                    continue
                ident, latex = res
                suffix = symbols.CYR_SUFFIX
                old = ident[:-len(suffix)] if ident.endswith(suffix) else ident
                old_g[old].add(latex)
                new_g[ident].add(latex)
                if re.search(r"[\u0400-\u04ff]", latex):
                    cyr += 1
                elif "\\" in latex:
                    greek += 1
                else:
                    lat += 1
    coll_old = sum(1 for v in old_g.values() if len(v) > 1)
    coll_new = sum(1 for v in new_g.values() if len(v) > 1)
    print(f"{doc:22} {cyr + greek + lat:>7} {cyr:>5} {greek:>5} {lat:>5} {coll_old:>8} {coll_new:>11}")
    tot = [a + b for a, b in zip(tot, [cyr + greek + lat, cyr, greek, lat, coll_old, coll_new])]
print(f"{'ИТОГО':22} {tot[0]:>7} {tot[1]:>5} {tot[2]:>5} {tot[3]:>5} {tot[4]:>8} {tot[5]:>11}")


документ                обозн.   кир  греч   лат  колл.до  колл.после
СП 22.13330.2016           213    19    70   124        5           0
СП 24.13330.2021           151    19    50    82        3           0
СП 30.13330.2020            33     9     5    19        0           0
СП 31.13330.2021            28    12     9     7        0           0
СП 45.13330.2017            40    13     5    22        0           0
ИТОГО                      465    72   139   254        8           0


### Разбор формул с кириллическими индексами

`restore_indices` подставляет латинский транслит обозначений (по словарю «где») вместо
OCR-мусора, после чего `latex2sympy2` парсит формулу. Кириллическое отображение при этом
сохраняется отдельно — sympy требует латинский идентификатор.


In [7]:
from normatives_pipeline.formulas.notation_restore import restore_indices
from normatives_pipeline.formulas.validation import canonical_latex

examples = [
    (r"P N \geq\frac{P_{pa6}+P_{\tau}} {1.4}", ["Рраб", "P\u03c4"]),
    (r"H_{3}=H_{3} a_{3} / 100 + 0.3", ["Нз", "аз"]),
    (r"v_{\phi}=v_{H} N_{\phi}/(N_{\phi}-N_{1})", []),
]
for latex, syms in examples:
    restored = restore_indices(latex, syms)
    parsed = not isinstance(canonical_latex(restored), str)
    print(("OK  " if parsed else "FAIL"), restored)


OK   PN\geq\frac{P_{rab}+P_{\tau}}{1.4}
OK   H_{z}=H_{z}a_{z}/100+0.3
OK   v_{\phi}=v_{H} N_{\phi}/(N_{\phi}-N_{1})
